In [1]:
!rm -r /kaggle/working/RCS_FUSIONDATA
!git clone https://github.com/Dlevinh755/RCS_FUSIONDATA.git

rm: cannot remove '/kaggle/working/RCS_FUSIONDATA': No such file or directory
Cloning into 'RCS_FUSIONDATA'...
remote: Enumerating objects: 39168, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 39168 (delta 72), reused 108 (delta 39), pack-reused 39024 (from 3)
Receiving objects: 100% (39168/39168), 387.02 MiB | 42.62 MiB/s, done.
Resolving deltas: 100% (218/218), done.
Updating files: 100% (38826/38826), done.


In [2]:
%%capture
!pip install -r /kaggle/working/RCS_FUSIONDATA/requirements.txt

In [3]:
# !python /kaggle/working/RCS_FUSIONDATA/prepare_data/prepare_amazon_product.py \
# --meta_link https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/metaFiles2/meta_Appliances.json.gz \
# --reviews_link https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/categoryFiles/Appliances.json.gz \
# --mode None \
# --json-parser parallel

In [4]:
!cd /kaggle/working/RCS_FUSIONDATA
!python /kaggle/working/RCS_FUSIONDATA/main.py \
--epochs 15 \
--batch_size 256 \
--lr 1e-4 \
--patience 3 \
--use_history 0

tokenizer_config.json: 100%|██████████████████| 25.0/25.0 [00:00<00:00, 189kB/s]
config.json: 100%|█████████████████████████████| 481/481 [00:00<00:00, 4.08MB/s]
vocab.json: 100%|████████████████████████████| 899k/899k [00:00<00:00, 6.28MB/s]
merges.txt: 100%|████████████████████████████| 456k/456k [00:00<00:00, 5.91MB/s]
tokenizer.json: 100%|██████████████████████| 1.36M/1.36M [00:00<00:00, 8.66MB/s]
⚡ Memory-efficient mode - processing on-the-fly
⚡ Memory-efficient mode - processing on-the-fly
⚡ Memory-efficient mode - processing on-the-fly
model.safetensors: 100%|██████████████████████| 499M/499M [00:01<00:00, 289MB/s]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/

In [5]:
# !python /kaggle/working/RCS_FUSIONDATA/find_similar_img.py \
# --query_image /kaggle/input/amazon-product/images/1397458135.jpg \
# --df_path /kaggle/working/RCS_FUSIONDATA/data/amazonproduct/val.csv \
# --k 10

In [ ]:
# Cell 1: Setup và Import
import sys
import os
import glob
import pandas as pd
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from torchvision import transforms
import warnings
warnings.filterwarnings('ignore')

# Add project path
PROJECT_PATH = '/kaggle/working/RCS_FUSIONDATA'
if os.path.isdir(PROJECT_PATH) and PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
    print("Đã thêm đường dẫn dự án vào sys.path.")

from model import CAMRec
from datahelper import AmazonReviewDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Helper: safe join for absolute/relative paths
def safe_join_path(base_dir, p):
    p = str(p)
    try:
        path_obj = Path(p)
        if path_obj.is_absolute():
            return str(path_obj)
    except Exception:
        pass
    cand = str(Path(base_dir) / p)
    return cand

# Cell 2: Define helper functions
def load_data(data_dir):
    """Load and combine train, val, test datasets"""
    print("=" * 60)
    print("📊 LOADING DATA...")
    print("=" * 60)
    
    data_dir = str(data_dir)
    if not data_dir.endswith('/'):
        data_dir += '/'
    
    train_df = pd.read_csv(data_dir + "train.csv")
    val_df = pd.read_csv(data_dir + "val.csv")
    test_df = pd.read_csv(data_dir + "test.csv")
    
    # Combine all data
    df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    
    # Ensure file_path is string (do not force prefix here; handle at usage time)
    df["file_path"] = df["file_path"].astype(str)
    
    print(f"✓ Total records: {len(df):,}")
    print(f"✓ Number of users: {df['reviewerID'].nunique():,}")
    print(f"✓ Number of items: {df['asin'].nunique():,}")
    print(f"✓ Columns: {df.columns.tolist()}\n")
    
    return df

def create_mappings(df):
    """Create user and item ID mappings"""
    print("=" * 60)
    print("🔑 CREATING MAPPINGS...")
    print("=" * 60)
    
    users = {u:i for i,u in enumerate(df['reviewerID'].astype(str).unique())}
    items = {a:i for i,a in enumerate(df['asin'].astype(str).unique())}
    
    # Reverse mapping
    idx_to_user = {i:u for u,i in users.items()}
    idx_to_item = {i:a for a,i in items.items()}
    
    print(f"✓ User mapping: {len(users):,} users")
    print(f"✓ Item mapping: {len(items):,} items\n")
    
    return users, items, idx_to_user, idx_to_item

def _discover_checkpoint(default_dirs, pattern='*.pth'):
    candidates = []
    for d in default_dirs:
        if not d: continue
        d = str(d)
        candidates.extend(glob.glob(os.path.join(d, '**', pattern), recursive=True))
    if not candidates:
        return None
    candidates = [(p, os.path.getmtime(p)) for p in candidates]
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[0][0]

def _clean_state_dict_keys(state_dict):
    cleaned = {}
    for k, v in state_dict.items():
        nk = k
        if nk.startswith('module.'):
            nk = nk[len('module.'): ]
        if nk.startswith('model.'):
            nk = nk[len('model.'): ]
        cleaned[nk] = v
    return cleaned

def load_model(model_path, n_users, n_items, device, use_pretrained=True):
    """Load trained recommendation model with robust checkpoint handling"""
    print("=" * 60)
    print("🤖 LOADING MODEL...")
    print("=" * 60)
    
    model = CAMRec(
        n_users=n_users, 
        n_items=n_items, 
        user_dim=128, 
        item_dim=128, 
        proj_dim=256, 
        heads=4
    ).to(device)
    
    ckpt_path = None
    if use_pretrained:
        if model_path and Path(model_path).exists():
            ckpt_path = model_path
        else:
            ckpt_path = _discover_checkpoint([
                '/kaggle/working',
                '/kaggle/working/RCS_FUSIONDATA',
                '/kaggle/working/RCS_FUSIONDATA/checkpoints',
                '/kaggle/working/RCS_FUSIONDATA/outputs',
                '/kaggle/working/RCS_FUSIONDATA/runs'
            ])
        if ckpt_path:
            print(f"Found checkpoint: {ckpt_path}")
        else:
            print("No checkpoint found. Using randomly initialized model.")
    
    if use_pretrained and ckpt_path and Path(ckpt_path).exists():
        try:
            checkpoint = torch.load(ckpt_path, map_location=device)
            
            if isinstance(checkpoint, dict) and ('state_dict' in checkpoint or 'model_state_dict' in checkpoint):
                state = checkpoint.get('state_dict', checkpoint.get('model_state_dict'))
            elif isinstance(checkpoint, dict):
                state = checkpoint
            else:
                state = checkpoint
            
            state = _clean_state_dict_keys(state)
            
            model_dict = model.state_dict()
            pretrained_dict = {}
            skipped = 0
            for k, v in state.items():
                if k in model_dict and model_dict[k].shape == v.shape:
                    pretrained_dict[k] = v
                else:
                    skipped += 1
            
            model_dict.update(pretrained_dict)
            model.load_state_dict(model_dict)
            
            print(f"\n✓ Loaded {len(pretrained_dict)}/{len(model_dict)} compatible parameters, skipped {skipped}")
            if skipped > 0:
                print("⚠ Some parameters were not loaded due to mismatch. Consider re-training for best results.\n")
        except Exception as e:
            print(f"⚠ Error loading checkpoint: {e}")
            print("  Using randomly initialized model instead\n")
    else:
        print("⚠ Using randomly initialized model (no checkpoint loaded)")
        print("  Note: Predictions will be random until model is trained\n")
    
    model.eval()
    print(f"✓ Model ready\n")
    
    return model

def choose_sample_user(df, users, min_history=7):
    """Select a random user with sufficient purchase history"""
    print("=" * 60)
    print("👤 SELECTING SAMPLE USER...")
    print("=" * 60)
    
    user_counts = df.groupby('reviewerID').size()
    eligible_users = user_counts[user_counts > min_history].index.tolist()
    
    if len(eligible_users) == 0:
        print(f"⚠ No users with history > {min_history}. Falling back to any user.")
        eligible_users = user_counts.index.tolist()
    
    print(f"✓ Eligible users: {len(eligible_users):,}")
    
    # Select random user
    selected_user = np.random.choice(eligible_users)
    selected_user_idx = users[str(selected_user)]
    
    print(f"✓ Selected user: {selected_user}")
    print(f"✓ User index: {selected_user_idx}")
    print(f"✓ Number of purchases: {int(user_counts[selected_user])}\n")
    
    return selected_user, selected_user_idx

def show_user_history(df, selected_user, base_dir="/kaggle/working/RCS_FUSIONDATA/"):
    """Display user's purchase history"""
    print("=" * 60)
    print("📜 USER PURCHASE HISTORY...")
    print("=" * 60)
    
    user_history = df[df['reviewerID'] == selected_user].copy()
    user_history = user_history.sort_values('overall', ascending=False)
    
    print(f"User: {selected_user}")
    print(f"{'='*80}")
    
    for idx, row in user_history.head(10).iterrows():
        print(f"\n{'─'*80}")
        print(f"Product ASIN: {row['asin']}")
        print(f"Title: {str(row.get('title', 'N/A'))[:70]}...")
        print(f"Rating: {float(row['overall']):.1f}/5.0")
        print(f"Image: {row['file_path']}")
    
    print(f"\n{'='*80}\n")
    
    # Plot images
    plot_user_products(user_history, selected_user, base_dir)
    
    return user_history

def plot_user_products(user_history, selected_user, base_dir):
    """Plot user's purchased products"""
    n_products = min(6, len(user_history))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(user_history.head(n_products).iterrows()):
        try:
            img_path = safe_join_path(base_dir, row['file_path'])
            img = Image.open(img_path).convert('RGB')
            axes[idx].imshow(img)
            title = str(row.get('title', 'N/A'))[:30]
            axes[idx].set_title(
                f"{title}...\nRating: {float(row['overall']):.1f}/5", 
                fontsize=9
            )
            axes[idx].axis('off')
        except Exception as e:
            axes[idx].text(0.5, 0.5, 'Image not found', ha='center', va='center')
            axes[idx].axis('off')
            print(f"⚠ Error loading image: {e}")
    
    # Hide unused axes
    for idx in range(n_products, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.suptitle(f'Products purchased by user {selected_user}', fontsize=14, y=1.02)
    plt.show()

def get_unpurchased_items(df, user_history):
    """Get list of items user hasn't purchased"""
    print("=" * 60)
    print("🔍 FILTERING UNPURCHASED ITEMS...")
    print("=" * 60)
    
    purchased_items = set(user_history['asin'].astype(str).unique())
    all_items = df['asin'].astype(str).unique()
    unpurchased_items = [item for item in all_items if item not in purchased_items]
    
    print(f"✓ Total products: {len(all_items):,}")
    print(f"✓ Already purchased: {len(purchased_items):,}")
    print(f"✓ Not purchased: {len(unpurchased_items):,}\n")
    
    return unpurchased_items

def _try_call_model(model, batch, device, use_amp):
    from torch import Tensor
    out = None
    err_msgs = []
    if use_amp and device == 'cuda':
        from torch.cuda.amp import autocast
        with autocast():
            try:
                out = model(batch)
            except Exception as e1:
                err_msgs.append(str(e1))
                try:
                    out = model(**batch)
                except Exception as e2:
                    err_msgs.append(str(e2))
    else:
        try:
            out = model(batch)
        except Exception as e1:
            err_msgs.append(str(e1))
            try:
                out = model(**batch)
            except Exception as e2:
                err_msgs.append(str(e2))
    
    if out is None:
        raise RuntimeError("Model forward failed: " + " | ".join(err_msgs))
    
    # Normalize output formats
    if isinstance(out, (list, tuple)) and len(out) > 0:
        out = out[0]
    if isinstance(out, dict):
        for k in ['pred', 'prediction', 'preds', 'logits', 'scores', 'rating', 'ratings']:
            if k in out:
                out = out[k]
                break
        else:
            out = next(iter(out.values()))
    return out

def predict_ratings(model, df, base_dir, unpurchased_items, selected_user, selected_user_idx, users, items, device, max_items=10000, batch_size=32, use_amp=True):
    """Predict ratings for unpurchased items with optimizations and model-compatibility"""
    print("=" * 60)
    print("🎯 PREDICTING RATINGS (OPTIMIZED)...")
    print("=" * 60)
    
    tok = AutoTokenizer.from_pretrained('roberta-base')
    
    img_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # Create user-item rating matrix for historical ratings (pre-compute once)
    pivot_df = pd.pivot_table(
        df,
        values='overall',
        index='reviewerID',
        columns='asin',
        aggfunc='mean',
        fill_value=0
    )
    
    # Pre-compute historical ratings tensor for this user
    hist_tensor_base = torch.zeros(len(items), dtype=torch.float32)
    if selected_user in pivot_df.index:
        historical_ratings = pivot_df.loc[selected_user].fillna(0.0)
        for asin, rating in historical_ratings.items():
            if asin in items:
                hist_tensor_base[items[asin]] = float(rating)
    
    predictions = []
    errors = []
    
    items_to_process = min(max_items, len(unpurchased_items))
    print(f"Processing {items_to_process:,} items in batches of {batch_size}...")
    print(f"Using AMP: {use_amp and device == 'cuda'}\n")
    
    # Process in batches
    with torch.no_grad():
        for batch_start in range(0, items_to_process, batch_size):
            batch_end = min(batch_start + batch_size, items_to_process)
            batch_items = unpurchased_items[batch_start:batch_end]
            
            batch_data = {
                'user_idx': [],
                'item_idx': [],
                'input_ids': [],
                'attention_mask': [],
                'image': [],
                'rating': [],
                'historical_ratings': [],
                'price': [],
                'meta': []
            }
            
            # Prepare batch
            for item_asin in batch_items:
                try:
                    # Get item info
                    item_rows = df[df['asin'].astype(str) == item_asin]
                    if len(item_rows) == 0:
                        continue
                    
                    item_info = item_rows.iloc[0]
                    item_idx = items[item_asin]
                    
                    # Prepare text input: prefer description, fallback to title
                    text = str(item_info.get('description', '') or '')
                    if not text or text.lower() == 'nan':
                        text = str(item_info.get('title', '') or '')
                    if not text:
                        text = "No content"
                    
                    tokens = tok(
                        text, 
                        padding='max_length', 
                        truncation=True, 
                        max_length=256, 
                        return_tensors='pt'
                    )
                    
                    # Prepare image input
                    img_path = safe_join_path(base_dir, item_info['file_path'])
                    if not Path(img_path).exists():
                        # skip if image missing
                        continue
                    
                    img = Image.open(img_path).convert('RGB')
                    img_tensor = img_transform(img)
                    
                    # Price
                    price = item_info.get('price', 0.0)
                    try:
                        price = float(price) if not pd.isna(price) else 0.0
                    except Exception:
                        price = 0.0
                    
                    # Add to batch
                    batch_data['user_idx'].append(selected_user_idx)
                    batch_data['item_idx'].append(item_idx)
                    batch_data['input_ids'].append(tokens['input_ids'].squeeze(0))
                    batch_data['attention_mask'].append(tokens['attention_mask'].squeeze(0))
                    batch_data['image'].append(img_tensor)
                    batch_data['historical_ratings'].append(hist_tensor_base.clone())
                    batch_data['price'].append(price)
                    batch_data['rating'].append(0.0)
                    batch_data['meta'].append({
                        'asin': item_asin,
                        'title': str(item_info.get('title', 'N/A')),
                        'price': price,
                        'file_path': str(item_info['file_path'])
                    })
                    
                except Exception as e:
                    errors.append(f"Error preparing {item_asin}: {str(e)}")
                    continue
            
            # Skip if batch is empty
            if len(batch_data['user_idx']) == 0:
                continue
            
            # Stack tensors
            try:
                batch = {
                    'user_idx': torch.tensor(batch_data['user_idx'], dtype=torch.long).to(device),
                    'item_idx': torch.tensor(batch_data['item_idx'], dtype=torch.long).to(device),
                    'input_ids': torch.stack(batch_data['input_ids']).to(device),
                    'attention_mask': torch.stack(batch_data['attention_mask']).to(device),
                    'image': torch.stack(batch_data['image']).to(device),
                    'rating': torch.zeros(len(batch_data['user_idx']), dtype=torch.float32).to(device),
                    'historical_ratings': torch.stack(batch_data['historical_ratings']).to(device),
                    'price': torch.tensor(batch_data['price'], dtype=torch.float32).to(device)
                }
                
                # Predict with flexible forward
                out = _try_call_model(model, batch, device, use_amp)
                preds = out
                if isinstance(preds, torch.Tensor):
                    preds = preds.detach().float().cpu().numpy()
                
                # Add predictions
                for idx, pred_rating in enumerate(preds):
                    if isinstance(pred_rating, (list, tuple, np.ndarray)):
                        pr = float(np.array(pred_rating).reshape(-1)[0])
                    else:
                        pr = float(pred_rating)
                    meta = batch_data['meta'][idx]
                    predictions.append({
                        'asin': meta['asin'],
                        'title': meta['title'],
                        'price': meta['price'],
                        'file_path': meta['file_path'],
                        'predicted_rating': pr
                    })
                
            except Exception as e:
                errors.append(f"Error predicting batch {batch_start}-{batch_end}: {str(e)}")
                continue
            
            # Progress update
            if (batch_end) % 100 == 0 or batch_end == items_to_process:
                print(f"  ✓ Processed {batch_end:,}/{items_to_process:,} items, {len(predictions):,} successful")
    
    print(f"\n✓ Successfully predicted for {len(predictions):,} products")
    
    if errors:
        print(f"⚠ {len(errors)} errors occurred. First 5:")
        for err in errors[:5]:
            print(f"  - {err}")
    
    print()
    return predictions

def get_recommendations(predictions, top_k=10):
    """Get top K recommendations"""
    print("=" * 60)
    print("⭐ TOP RECOMMENDATIONS...")
    print("=" * 60)
    
    recommendations = sorted(
        predictions, 
        key=lambda x: x['predicted_rating'], 
        reverse=True
    )[:top_k]
    
    print(f"\nTOP {top_k} RECOMMENDED PRODUCTS:")
    print(f"{'='*80}\n")
    
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. Product ID: {rec['asin']}")
        print(f"   Title: {rec['title'][:70]}...")
        print(f"   Predicted Rating: {rec['predicted_rating']:.2f}/5.0")
        print()
    
    return recommendations

def plot_recommendations(recommendations, selected_user, base_dir="/kaggle/working/RCS_FUSIONDATA/"):
    """Plot recommended products"""
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    
    for idx, rec in enumerate(recommendations):
        try:
            img_path = safe_join_path(base_dir, rec['file_path'])
            img = Image.open(img_path).convert('RGB')
            axes[idx].imshow(img)
            title = rec['title'][:25]
            axes[idx].set_title(
                f"#{idx+1}: {title}...\n"
                f"Pred: {rec['predicted_rating']:.2f}/5.0",
                fontsize=9
            )
            axes[idx].axis('off')
        except Exception as e:
            axes[idx].text(0.5, 0.5, 'Image not found', ha='center', va='center')
            axes[idx].axis('off')
            print(f"⚠ Error loading image: {e}")
    
    plt.tight_layout()
    plt.suptitle(
        f'TOP 10 Recommended products for user {selected_user}', 
        fontsize=16, 
        y=1.02
    )
    plt.show()

# Cell 3: Main Pipeline Function
def run_recommendation_pipeline(
    data_dir="/kaggle/working/RCS_FUSIONDATA/data/amazonproduct/",
    model_path=None,
    base_dir="/kaggle/working/RCS_FUSIONDATA/",
    min_history=5,
    max_predict_items=2000,
    top_k=10,
    use_pretrained=True,
    batch_size=256,  # Batch size for inference
    use_amp=True    # Use automatic mixed precision
):
    print("\n" + "="*60)
    print("🚀 STARTING RECOMMENDATION PIPELINE")
    print("="*60 + "\n")
    
    try:
        # Step 1: Load data
        df = load_data(data_dir)
        
        # Step 2: Create mappings
        users, items, idx_to_user, idx_to_item = create_mappings(df)
        
        # Step 3: Load model
        model = load_model(model_path, len(users), len(items), device, use_pretrained)
        
        # Step 4: Select sample user
        selected_user, selected_user_idx = choose_sample_user(df, users, min_history)
        
        # Step 5: Show user history
        user_history = show_user_history(df, selected_user, base_dir)
        
        # Step 6: Get unpurchased items
        unpurchased_items = get_unpurchased_items(df, user_history)
        
        # Step 7: Predict ratings (optimized with batching)
        predictions = predict_ratings(
            model, df, base_dir, unpurchased_items, 
            selected_user, selected_user_idx, users, items, device, 
            max_predict_items,
            batch_size=batch_size,
            use_amp=use_amp
        )
        
        # Step 8: Get recommendations
        recommendations = get_recommendations(predictions, top_k)
        
        # Step 9: Plot recommendations
        plot_recommendations(recommendations, selected_user, base_dir)
        
        print("\n" + "="*60)
        print("✅ PIPELINE COMPLETED SUCCESSFULLY")
        print("="*60 + "\n")
    except Exception as e:
        print(f"⚠ Error in pipeline: {e}")
        print("  Please check the logs for more details.")

ImportError: cannot import name 'is_directory' from 'PIL._util' (/usr/local/lib/python3.11/dist-packages/PIL/_util.py)

In [ ]:
run_recommendation_pipeline(use_pretrained=True)